# 사용자 지정 컨테이너 - Go

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | Harness - 사용자 지정 컨테이너(Go) + ExecuteCommand |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

**학습 내용:**
- 에이전트가 Go 프로그램을 컴파일하고 실행할 수 있도록 Harness에 **Go** 컨테이너 연결
- **ExecuteCommand**를 사용하여 Go 툴체인을 확인하고 VM의 결과물 점검

**Go를 사용하는 이유**
Go는 하나의 정적 바이너리로 컴파일되므로 런타임 종속성, `node_modules`, virtualenv가 필요하지 않습니다. 따라서 CLI 도구, 마이크로서비스 또는 시스템 유틸리티를 만드는 에이전트에 적합합니다. 에이전트는 격리된 microVM 안에서 `.go` 파일을 작성하고 `go build`를 실행한 뒤 바이너리를 실행합니다.

> 함께 보기: [Node.js 사용자 지정 컨테이너](01_custom_container_node.ipynb) 및 [CLI 스크립트](02_custom_container_cli.py)(모든 컨테이너 이미지에서 작동)

## 0단계: 설정

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## 1단계: Harness 생성 + Go 컨테이너 연결

하나의 흐름에서 Harness를 생성한 뒤 즉시 Go 컨테이너 이미지를 사용하도록 업데이트합니다.

In [ ]:
HARNESS_NAME = f"GoContainer_{uuid.uuid4().hex[:8]}"
CONTAINER_URI = "public.ecr.aws/docker/library/golang:1.24"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")

for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        break
    time.sleep(5)

print(f"\nAttaching container: {CONTAINER_URI}")
control.update_harness(
    harnessId=harness_id,
    environmentArtifact={"optionalValue": {"containerConfiguration": {"containerUri": CONTAINER_URI}}},
    systemPrompt=[
        {
            "text": "You are a helpful coding assistant. You have access to a full Go toolchain. When asked to write and run code, save .go files, use 'go run' or 'go build', and execute the resulting binary."
        }
    ],
)

for i in range(24):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness ready with Go container")
        break
    time.sleep(5)

## 2단계: 호출 - Go 프로그램 작성, 빌드 및 실행

에이전트에 Go로 HTTP 서버를 작성하고 컴파일한 뒤 테스트하도록 요청합니다. 소스 작성, `go build`, 바이너리 실행으로 이어지는 전체 Go 워크플로를 보여 줍니다.

In [ ]:
import uuid

session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Write a Go HTTP server that listens on port 3000 and returns a JSON response "
                        "with the current time, Go version, OS, architecture, and number of CPUs. "
                        "Initialize a Go module at /tmp/goserver, save the code as main.go, build it "
                        "into a binary called 'goserver', then test it: start the binary in the background, "
                        "curl localhost:3000, and kill the server. Show me the curl output."
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## 3단계: ExecuteCommand - Go 환경 확인

In [ ]:
def run_command(command: str):
    """에이전트 VM에서 명령을 실행하고 출력을 표시합니다."""
    print(f"$ {command}")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": command},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk:
                d = chunk["contentDelta"]
                if "stdout" in d:
                    print(d["stdout"], end="", flush=True)
                if "stderr" in d:
                    print(d["stderr"], end="", flush=True)
            elif "contentStop" in chunk:
                print(f"\n[exit: {chunk['contentStop']['exitCode']}]")
    print()

In [ ]:
run_command("go version")
run_command("go env GOROOT GOPATH GOARCH GOOS")

### 생성된 소스 및 바이너리 점검

In [ ]:
run_command("cat /tmp/goserver/main.go")
run_command("ls -lh /tmp/goserver/")

### 컴파일된 바이너리 직접 다시 실행

In [ ]:
# ExecuteCommand로 컴파일된 바이너리를 시작하고 curl로 호출한 뒤 종료
run_command(
    "cd /tmp/goserver && ./goserver & sleep 1 && curl -s http://localhost:3000 | python3 -m json.tool && kill %1 2>/dev/null"
)

## 4단계: 추가 실습 - Linux AMD64용 크로스 컴파일

Go에서는 크로스 컴파일이 간단합니다. 에이전트에 같은 서버를 `linux/amd64`용으로 크로스 컴파일하고 바이너리를 확인하도록 요청해 보겠습니다.

In [ ]:
# 같은 세션에서는 상태가 유지됨
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Cross-compile the Go server at /tmp/goserver for linux/amd64 as /tmp/goserver/goserver-amd64. "
                        "Then use 'file' to show the binary type of both the original and the cross-compiled binary."
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## 리소스 정리

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
# IAM 역할 삭제(다른 노트북에서 사용할 경우 유지 가능)
delete_harness_role()